# i adapt his lecture file to be for our project
# and by adapt i mean i change it from cartpole to acrobot and it just works

This notebook follows the PyTorch DQN tutorial structure and adapts it for lecture use.

Reference tutorial:
- [PyTorch DQN Tutorial](https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html)

In this notebook, you will:
- define a DQN network,
- train with replay memory and a target network,
- evaluate a trained policy,
- save rollout video output.

## 1) Install dependency

Install Gymnasium classic-control environments (CartPole). Run once per fresh runtime.

In [1]:
#!pip3 install gymnasium[classic_control]
#!pip3 install gymnasium[other]

## 2) Imports

Load Python, plotting, and PyTorch packages used for environment setup, network training, and optimization.

In [2]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


## DQN Algorithm

Our environment is deterministic, so all equations presented here are also formulated deterministically for the sake of simplicity. In the reinforcement learning literature, they would also contain expectations over stochastic transitions in the environment.

Our aim will be to train a policy that tries to maximize the discounted, cumulative reward

$$
R_{t_0} = \sum_{t=t_0}^{\infty} \gamma^{t-t_0} r_t
$$

where $R_{t_0}$ is also known as the return. The discount, $\gamma$, should be a constant between $0$ and $1$ that ensures the sum converges. A lower $\gamma$ makes rewards from the uncertain far future less important for our agent than the ones in the near future that it can be fairly confident about. It also encourages agents to collect reward closer in time than equivalent rewards that are temporally far away in the future.

The main idea behind Q-learning is that if we had a function

$$
Q^*: State \times Action \rightarrow \mathbb{R}
$$

that could tell us what our return would be if we were to take an action in a given state, then we could easily construct a policy that maximizes our rewards:

$$
\pi^*(s) = \arg\max_a Q^*(s,a)
$$

However, we do not know everything about the world, so we do not have access to $Q^*$. But, since neural networks are universal function approximators, we can simply create one and train it to resemble $Q^*$.

For our training update rule, we will use a fact that every $Q$ function for some policy obeys the Bellman equation:

$$
Q^\pi(s,a) = r + \gamma Q^\pi(s', \pi(s'))
$$

The difference between the two sides of the equality is known as the temporal difference error, $\delta$:

$$
\delta = Q(s,a) - \left(r + \gamma \max_{a'} Q(s',a')\right)
$$

To minimize this error, we will use the Huber loss. The Huber loss acts like the mean squared error when the error is small, but like the mean absolute error when the error is large. This makes it more robust to outliers when the estimates of $Q$ are very noisy. We calculate this over a batch of transitions, $B$, sampled from replay memory:

$$
\mathcal{L} = \frac{1}{|B|}\sum_{(s,a,s',r)\in B} \mathcal{L}(\delta)
$$

where

$$
\mathcal{L}(\delta) =
\begin{cases}
\frac{1}{2}\delta^2 & \text{for } |\delta| \le 1, \\
|\delta| - \frac{1}{2} & \text{otherwise.}
\end{cases}
$$

## 3) Environment + hyperparameters + training plot helper

Initialize CartPole, detect compute device, define DQN hyperparameters, and create `plot_durations()` for training-progress visualization.

In [3]:
name = "Acrobot-v1"
env = gym.make(name)

# set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

# if GPU is to be used
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

# Hyperparameters from the PyTorch tutorial
BATCH_SIZE = 28
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 2500
TAU = 0.005
LR = 3e-4

# Get number of actions from gym action space
n_actions = env.action_space.n
# Get the number of state observations
state, info = env.reset()
n_observations = len(state)

episode_durations = []


def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

## 4) Define Q-network (DQN)

Create a small feed-forward network that maps a CartPole state vector to Q-values for each action.

- input: state (4 features), Cart Position, Cart Velocity, Pole Angle, Pole Angular Velocity
- output: Q-value per action (left/right)
- hidden layers use ReLU activations

In [4]:
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))


class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)


class DQN(nn.Module):

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)


## 5) Initialize networks + optimizer + replay memory

### Two-network view
- **policy_net**: learns every optimization step.
- **target_net**: provides stable bootstrap targets (updated periodically).


Create `policy_net`, `target_net`, `AdamW` optimizer, tutorial replay memory, and `steps_done` counter.


In [5]:
policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)

steps_done = 0

print("Initialization already completed in Cell 13.")


Initialization already completed in Cell 13.


## 6) Epsilon-greedy action selection

Use per-step exponential epsilon decay (`EPS_START`, `EPS_END`, `EPS_DECAY`) and select either random action or greedy `policy_net` action.

In [6]:
def select_action(state):
    global steps_done
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1.0 * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        with torch.no_grad():
            return policy_net(state).max(1).indices.view(1, 1)
    return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)


## 7) Optimize step
Sample a batch from replay memory, compute state-action values and target values with `target_net`, use Huber loss, and apply gradient clipping.

In [7]:
def optimize_model():
    if len(memory) < BATCH_SIZE:
        return
    transitions = memory.sample(BATCH_SIZE)
    batch = Transition(*zip(*transitions))

    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    state_action_values = policy_net(state_batch).gather(1, action_batch)

    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values

    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()


## 8) Training loop

Run the official loop pattern: environment reset, action selection, transition storage, optimize step, and soft target network update each step (`TAU`).

In [8]:
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 500
else:
    num_episodes = 50

for i_episode in range(num_episodes):
    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    for t in count():
        action = select_action(state)
        observation, reward, terminated, truncated, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        done = terminated or truncated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        memory.push(state, action, next_state, reward)

        state = next_state

        optimize_model()

        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)
        target_net.load_state_dict(target_net_state_dict)

        if done:
            episode_durations.append(t + 1)
            plot_durations()
            break

print('Complete')
plot_durations(show_result=True)
plt.ioff()
plt.show()

KeyboardInterrupt: 

## 9) Run trained policy with live frame updates

Use greedy policy (`argmax Q`) to roll out one episode and display rendered frames inline (useful for qualitative behavior checks).

In [ ]:
import gymnasium as gym
import torch
import matplotlib.pyplot as plt
from IPython import display

# Ensure we use the correct render mode for Colab
env = gym.make(name, render_mode="rgb_array")

state, _ = env.reset()
done = False

# Initialize the display
plt.figure(figsize=(5, 4))
img = plt.imshow(env.render())
plt.axis('off')

while not done:
    with torch.no_grad():
        # Convert state to tensor and move to device
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        print(policy_net(state_t))
        action = policy_net(state_t).argmax().item()

    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    # Render and update the plot
    img.set_data(env.render())
    display.display(plt.gcf())
    display.clear_output(wait=True)

env.close()

## 10) Save rollout video

Record one greedy-policy episode and save rendered frames to a video file in the `videos/` directory.

In [ ]:
import gymnasium as gym
from gymnasium.utils.save_video import save_video
import torch

# Set up the environment with rgb_array for recording
env = gym.make(name, render_mode="rgb_array_list")

state, _ = env.reset()
step_starting_index = 0
episode_index = 0
done = False

while not done:
    with torch.no_grad():
        # Convert state to tensor and move to the same device as the policy network
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        action = policy_net(state_t).argmax().item()

    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

# Save the recorded frames as a video
save_video(
    env.render(),
    "videos",
    fps=30,
    episode_index=episode_index,
    name_prefix="cartpole-run"
)

env.close()

print("Video saved to the 'videos' folder.")